In [1]:
import yaml
import numpy as np
import geopandas as gpd
import rioxarray
from pathlib import Path
from shapely.geometry import LineString

# load config
config_path = Path("..") / "config.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# load power lines
lines = gpd.read_file(Path("..") / "data" / "raw" / "power_lines.gpkg")

# load index rasters
processed_dir = Path("..") / "data" / "processed"
ndvi = rioxarray.open_rasterio(processed_dir / "ndvi.tif", masked=True).squeeze()
ndwi = rioxarray.open_rasterio(processed_dir / "ndwi.tif", masked=True).squeeze()
slope = rioxarray.open_rasterio(processed_dir / "slope.tif", masked=True).squeeze()

print(f"Power lines: {len(lines)} features, CRS: {lines.crs}")
print(f"NDVI CRS: {ndvi.rio.crs}")
print(f"Slope CRS: {slope.rio.crs}")


Power lines: 140 features, CRS: EPSG:4326
NDVI CRS: EPSG:32634
Slope CRS: EPSG:3067


In [2]:
# reproject slope from EPSG:3067 to match NDVI exactly
# — same CRS (EPSG:32634), same resolution (10m), same pixel grid
slope_10m = slope.rio.reproject_match(ndvi)

print(f"Slope CRS after: {slope_10m.rio.crs}")
print(f"Slope shape after: {slope_10m.shape}")
print(f"NDVI shape: {ndvi.shape}")


Slope CRS after: EPSG:32634
Slope shape after: (4591, 3384)
NDVI shape: (4591, 3384)


In [3]:
def split_line(line, segment_length_m):
    """Split a LineString into segments of roughly segment_length_m metres."""
    total_length = line.length
    segments = []
    distance = 0.0
    while distance < total_length:
        start = line.interpolate(distance)
        end = line.interpolate(min(distance + segment_length_m, total_length))
        segments.append(LineString([start, end]))
        distance += segment_length_m
    return segments

# test on the first line — reproject to metric CRS first
lines_metric = lines.to_crs(epsg=3067)
test_segments = split_line(lines_metric.geometry.iloc[0], segment_length_m=100)

print(f"First line length: {lines_metric.geometry.iloc[0].length:.0f}m")
print(f"Number of 100m segments: {len(test_segments)}")
print(f"Last segment length: {test_segments[-1].length:.1f}m")


First line length: 1096m
Number of 100m segments: 11
Last segment length: 92.9m


In [4]:
segment_rows = []

for _, row in lines_metric.iterrows():
    segs = split_line(row.geometry, segment_length_m=100)
    for seg in segs:
        segment_rows.append({
            "geometry": seg,
            "voltage": row.get("voltage"),
            "operator": row.get("operator"),
            "ref": row.get("ref"),
        })

segments = gpd.GeoDataFrame(segment_rows, crs="EPSG:3067")

# reproject back to EPSG:4326 for compatibility with rasters
segments = segments.to_crs(epsg=4326)

print(f"Total segments: {len(segments)}")
print(segments.head(3))


Total segments: 3554
                                           geometry voltage operator  ref
0        LINESTRING (23.765 61.3, 23.76414 61.3008)  110000      NaN  NaN
1  LINESTRING (23.76414 61.3008, 23.76317 61.30156)  110000      NaN  NaN
2  LINESTRING (23.76317 61.30156, 23.76212 61.3023)  110000      NaN  NaN


In [5]:
# reproject segments to raster CRS (EPSG:32634) for sampling
segs_utm = segments.to_crs(ndvi.rio.crs)

ndvi_vals, ndwi_vals, slope_vals = [], [], []

for geom in segs_utm.geometry:
    # midpoint of each segment
    mid = geom.interpolate(0.5, normalized=True)
    ndvi_vals.append(float(ndvi.sel(x=mid.x, y=mid.y, method="nearest")))
    ndwi_vals.append(float(ndwi.sel(x=mid.x, y=mid.y, method="nearest")))
    slope_vals.append(float(slope_10m.sel(x=mid.x, y=mid.y, method="nearest")))

segments["ndvi"] = ndvi_vals
segments["ndwi"] = ndwi_vals
segments["slope"] = slope_vals

print(segments[["ndvi", "ndwi", "slope"]].describe().round(3))


           ndvi      ndwi     slope
count  3554.000  3554.000  3554.000
mean      0.447    -0.417     4.901
std       0.137     0.114     3.704
min      -0.046    -0.648     0.000
25%       0.382    -0.495     2.265
50%       0.466    -0.431     4.084
75%       0.547    -0.371     6.709
max       0.707     0.110    61.091


In [6]:
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

# normalize each index to [0, 1]
segments["ndvi_norm"] = minmax(segments["ndvi"])
segments["ndwi_norm"] = minmax(segments["ndwi"])
segments["slope_norm"] = minmax(segments["slope"])

# weighted risk score from config
w = config["risk"]["weights"]
segments["risk_score"] = (
    w["ndvi"]  * segments["ndvi_norm"] +
    w["ndwi"]  * segments["ndwi_norm"] +
    w["slope"] * segments["slope_norm"]
)

print(segments["risk_score"].describe().round(3))


count    3554.000
mean        0.435
std         0.050
min         0.279
25%         0.410
50%         0.442
75%         0.471
max         0.674
Name: risk_score, dtype: float64


In [7]:
out_dir = Path("..") / "data" / "processed"

# save as GeoPackage (for GIS tools like QGIS)
segments.to_file(out_dir / "segments_risk.gpkg", driver="GPKG")

# save as GeoJSON (for web maps)
segments.to_file(out_dir / "segments_risk.geojson", driver="GeoJSON")

print(f"Saved {len(segments)} segments with risk scores")
print(segments[["voltage", "risk_score"]].head())


Saved 3554 segments with risk scores
  voltage  risk_score
0  110000    0.451146
1  110000    0.427629
2  110000    0.457032
3  110000    0.483037
4  110000    0.470641
